#Importando o ultralytics, crew ai e configurando chave de APi

In [1]:
!pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 51.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 22.0 MB/s eta 0:00:00


In [2]:
!pip install crewai litellm --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 6.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 379.2/379.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 113.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.6/19.6 MB 114.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.5/119.5 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.6/65.6 kB 6.3 MB/s e

In [10]:
import os
os.environ["GEMINI_API_KEY"] = "GEMINI_API_KEY"

#Importando as bibliotecas, configurando e iniciando o rastreamento

In [12]:
import cv2
import json
from ultralytics import solutions

# Abrir vídeo
#https://pixabay.com/pt/videos/pessoas-com%C3%A9rcio-fazer-compras-6387/
cap = cv2.VideoCapture("videorastreio2.mp4")
assert cap.isOpened(), "Erro ao abrir vídeo"

w, h, fps = (int(cap.get(x)) for x in (
    cv2.CAP_PROP_FRAME_WIDTH, cv2.CAP_PROP_FRAME_HEIGHT, cv2.CAP_PROP_FPS))

line_points = [(880, 400), (1080, 400)]  # Linha para contagem

# Criar objeto contador
counter = solutions.ObjectCounter(model="yolov8n.pt", classes=[0])
counter.region = line_points  # Definir região de contagem

video_writer = cv2.VideoWriter(
    "counting-output.mp4",
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps, (w, h)
)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = counter.process(frame)

    # Salvar frame anotado no vídeo
    video_writer.write(results.plot_im)

    # Exibir contagens na tela (opcional)
    print(f"In count: {counter.in_count}, Out count: {counter.out_count}")

cap.release()
video_writer.release()

# Preparar dados para salvar em arquivo texto JSON
output_data = {
    "in_count": counter.in_count,
    "out_count": counter.out_count,
    "classwise_count": counter.classwise_count,
    "total_tracks": results.total_tracks
}

# Salvar dados em arquivo .txt (formato JSON legível)
with open("counting_output.txt", "w") as f:
    json.dump(output_data, f, indent=4)

print("Dados salvos em counting_output.txt")


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics Solutions: ✅ {'source': None, 'model': 'yolov8n.pt', 'classes': [0], 'show_conf': True, 'show_labels': True, 'region': None, 'colormap': 21, 'show_in': True, 'show_out': True, 'up_angle': 145.0, 'down_angle': 90, 'kpts': [6, 8, 10], 'analytics_type': 'line', 'figsize': (12.8, 7.2), 'blur_ratio': 0.5, 'vision_point': (20, 20), 'crop_dir': 'cropped-detections', 'json_file': None, 'line_width': 2, 'records': 5, 'fps': 30.0, 'max_hist': 5, 'meter_per_pixel': 0.05, 'max_speed': 120, 'show': False, 'iou': 0.7, 'conf': 0.25, 'device': None, 'max_det': 300, 'half': False, 'tracker': 'botsort.yaml', 'verbose': True, 'data': 'images'}


WARNING ⚠️ Environment does not support cv2.imshow() or PIL Image.show()



requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...

requirements: AutoUpdate success ✅ 0.3s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

In count: 0, Out count: 0
In count: 0, Out count: 0
In count: 0, Out count: 0
In count: 0, Out count: 0
In count: 0, Out count: 0
In count: 0, Out count: 0
In count: 0, Out count: 0
In count: 0, Out count: 0
In count: 0, Out count: 0
In count: 0, Out count: 0
In count: 0, Out count: 0
In count: 0, Out count: 0
In count: 0, Out count: 0
In count: 0, Out count: 0
In count: 0, Out count: 0
In count: 0, Out count: 0
In count: 0, Out count: 0
In count: 0, Out count: 0
In count: 0, Out count: 0
In count: 0, Out count: 0
In count: 0, Out count: 0
In count: 0, Out count: 0
In count: 0, Out count: 0
In count: 0, Out count: 0
In count: 0, Out count: 0
In count: 0, Out count: 0
In count: 0, Out count: 0
In count: 0, Out count: 0
In count: 0, Out count: 0
In count: 0, Out count: 0
In 

In [22]:
import os
from crewai import LLM, Agent, Task, Crew

# Caminho do TXT
caminho_txt = "/content/counting_output.txt"

# Lendo o conteúdo do TXT
with open(caminho_txt, "r", encoding="utf-8") as f:
    conteudo_txt = f.read()

# Criando o LLM do Gemini
gemini_llm = LLM(
    model="gemini/gemini-2.0-flash",  # ou gemini-1.5-pro
    temperature=0.7,
    api_key=os.environ["GEMINI_API_KEY"]
)

# Criando um agente de exemplo
agent = Agent(
    role="Analista",
    goal="Gerar relatórios interpretativos a partir de dados de contagem de pessoas.",
    backstory="Especialista em análise de vídeo monitoramento e comportamento humano.",
    verbose=True,
    llm=gemini_llm
)

# Criando uma tarefa de exemplo
task = Task(
    description="Depois de uma analise de rastreamento gerou os dados de que 3 pessoas ultrapassaram uma area delimitada e 1 pessoa saiu, foram registradas com precisão 31 pessoas e quero que gere um relatório descritivo, claro e objetivo com os dados de rastreamento que estão no arquivo.",
    expected_output="Relatório textual interpretativo.",
    agent=agent,
    input=conteudo_txt
)

# Executando
crew = Crew(agents=[agent], tasks=[task], verbose=True)
resultado = crew.kickoff()
print(resultado)

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 50334cad-15b3-4130-a1cd-fc2455d08b28                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analista                                                                                                │
│                                                                                                                 │
│  Task: Depois de uma analise de rastreamento gerou os dados de que 3 pessoas ultrapassaram uma area delimitada  │
│  e 1 pessoa saiu, foram registradas com precisão 31 pessoas e quero que gere um relatório descritivo, claro e   │
│  objetivo com os dados de rastreamento que estão no arquivo.                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analista                                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Relatório de Análise de Contagem de Pessoas**                                                                │
│                                                                                                                 │
│  **Data:** [Inserir Data]                                                                                       │
│  **Hora:** [Inserir Hora, se aplicável, ou período da análise]                                                  │
│  **Local:** [Inserir Localização da área delimitada]                                                            │
│                                                                                                                 │
│  **Resumo:**                                                                                                    │
│                                                                                                                 │
│  Este relatório apresenta uma análise dos dados de contagem de pessoas em uma área delimitada específica. Os    │
│  dados foram coletados por meio de um sistema de rastreamento de vídeo e processados para fornecer informações  │
│  sobre o fluxo de pessoas na área monitorada.                                                                   │
│                                                                                                                 │
│  **Resultados:**                                                                                                │
│                                                                                                                 │
│  *   **Entradas:** Um total de 3 pessoas foram registradas entrando na área delimitada.                         │
│  *   **Saídas:** Um total de 1 pessoa foi registrada saindo da área delimitada.                                 │
│  *   **Contagem Total:** O sistema registrou com precisão a presença de 31 pessoas na área monitorada durante   │
│  o período da análise.                                                                                          │
│                                                                                                                 │
│  **Interpretação:**                                                                                             │
│                                                                                                                 │
│  Os dados indicam um fluxo de pessoas para dentro da área delimitada, com um número maior de entradas (3) em    │
│  comparação com as saídas (1). A contagem total de 31 pessoas sugere o nível de ocupação da área durante o      │
│  período analisado.                                                                                             │
│                                                                                                                 │
│  **Considerações:**                                                                                             │
│                                                                                                                 │
│  *   É importante considerar o contexto da área delimitada para interpretar os dados com precisão. Por          │
│  exemplo, se a área for uma loja, as entradas e saídas podem indicar o número de clientes que entraram e        │
│  saíram da loja.                                       

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 4faf0566-012b-4b34-9d98-ce7470c3ae32                                                                     │
│  Agent: Analista                                                                                                │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 50334cad-15b3-4130-a1cd-fc2455d08b28                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: **Relatório de Análise de Contagem de Pessoas**                                                  │
│                                                                                                                 │
│  **Data:** [Inserir Data]                                                                                       │
│  **Hora:** [Inserir Hora, se aplicável, ou período da análise]                                                  │
│  **Local:** [Inserir Localização da área delimitada]                                                            │
│                                                                                                                 │
│  **Resumo:**                                                                                                    │
│                                                                                                                 │
│  Este relatório apresenta uma análise dos dados de contagem de pessoas em uma área delimitada específica. Os    │
│  dados foram coletados por meio de um sistema de rastreamento de vídeo e processados para fornecer informações  │
│  sobre o fluxo de pessoas na área monitorada.                                                                   │
│                                                                                                                 │
│  **Resultados:**                                                                                                │
│                                                                                                                 │
│  *   **Entradas:** Um total de 3 pessoas foram registradas entrando na área delimitada.                         │
│  *   **Saídas:** Um total de 1 pessoa foi registrada saindo da área delimitada.                                 │
│  *   **Contagem Total:** O sistema registrou com precisão a presença de 31 pessoas na área monitorada durante   │
│  o período da análise.                                                                                          │
│                                                                                                                 │
│  **Interpretação:**                                                                                             │
│                                                                                                                 │
│  Os dados indicam um fluxo de pessoas para dentro da área delimitada, com um número maior de entradas (3) em    │
│  comparação com as saídas (1). A contagem total de 31 pessoas sugere o nível de ocupação da área durante o      │
│  período analisado.                                                                                             │
│                                                                                                                 │
│  **Considerações:**                                                                                             │
│                                                                                                                 │
│  *   É importante considerar o contexto da área delimitada para interpretar os dados com precisão. Por          │
│  exemplo, se a área for uma loja, as entradas e saídas

**Relatório de Análise de Contagem de Pessoas**

**Data:** [Inserir Data]
**Hora:** [Inserir Hora, se aplicável, ou período da análise]
**Local:** [Inserir Localização da área delimitada]

**Resumo:**

Este relatório apresenta uma análise dos dados de contagem de pessoas em uma área delimitada específica. Os dados foram coletados por meio de um sistema de rastreamento de vídeo e processados para fornecer informações sobre o fluxo de pessoas na área monitorada.

**Resultados:**

*   **Entradas:** Um total de 3 pessoas foram registradas entrando na área delimitada.
*   **Saídas:** Um total de 1 pessoa foi registrada saindo da área delimitada.
*   **Contagem Total:** O sistema registrou com precisão a presença de 31 pessoas na área monitorada durante o período da análise.

**Interpretação:**

Os dados indicam um fluxo de pessoas para dentro da área delimitada, com um número maior de entradas (3) em comparação com as saídas (1). A contagem total de 31 pessoas sugere o nível de ocupação da 

╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: f72e876b-9d3b-4c07-82f3-35af7d414cf9                                                                       │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analista                                                                                                │
│                                                                                                                 │
│  Task: Analise o conteudo em formato json:                                                                      │
│                                                                                                                 │
│  {conteudo_txt} e gere um relatório descritivo, claro e objetivo com os dados de rastreamento.                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Output()

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Analista                                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ```text                                                                                                        │
│  Relatório de Análise de Contagem de Pessoas                                                                    │
│                                                                                                                 │
│  Este relatório apresenta uma análise dos dados de contagem de pessoas obtidos a partir de um sistema de vídeo  │
│  monitoramento. O objetivo é fornecer uma visão geral do fluxo de pessoas em um determinado local e período,    │
│  identificando padrões e tendências relevantes.                                                                 │
│                                                                                                                 │
│  Metodologia                                                                                                    │
│                                                                                                                 │
│  Os dados foram coletados por meio de câmeras de vídeo equipadas com algoritmos de contagem de pessoas. As      │
│  informações foram registradas em formato JSON e incluem o número de pessoas que entraram e saíram de uma área  │
│  específica, bem como o timestamp correspondente. A análise foi realizada utilizando técnicas estatísticas      │
│  descritivas para resumir e interpretar os dados.                                                               │
│                                                                                                                 │
│  Análise Geral                                                                                                  │
│                                                                                                                 │
│  A análise dos dados revela o seguinte:                                                                         │
│                                                                                                                 │
│  *   **Fluxo Total:** O número total de pessoas que entraram na área monitorada durante o período analisado     │
│  foi de [inserir número total de entradas]. Em contrapartida, o número total de pessoas que saíram foi de       │
│  [inserir número total de saídas]. A diferença entre entradas e saídas pode indicar uma tendência de aumento    │
│  ou diminuição da população na área.                                                                            │
│                                                                                                                 │
│  *   **Picos de Movimentação:** Foram identificados horários de pico de movimentação, nos quais o número de     │
│  entradas e saídas foi significativamente maior do que a média. Esses picos ocorreram em [inserir horários de   │
│  pico]. A identificação desses horários pode ser útil para otimizar a alocação de recursos e melhorar a gestão  │
│  do espaço.                                                                                                     │
│                                                                                                                 │
│  *   **Vale de Movimentação:** Similarmente, foram identificados horários de baixo fluxo de pessoas, nos quais  │
│  o número de entradas e saídas foi significativamente m

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: e00b803a-1e1d-48dc-b89e-0b69afacc749                                                                     │
│  Agent: Analista                                                                                                │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: f72e876b-9d3b-4c07-82f3-35af7d414cf9                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: ```text                                                                                          │
│  Relatório de Análise de Contagem de Pessoas                                                                    │
│                                                                                                                 │
│  Este relatório apresenta uma análise dos dados de contagem de pessoas obtidos a partir de um sistema de vídeo  │
│  monitoramento. O objetivo é fornecer uma visão geral do fluxo de pessoas em um determinado local e período,    │
│  identificando padrões e tendências relevantes.                                                                 │
│                                                                                                                 │
│  Metodologia                                                                                                    │
│                                                                                                                 │
│  Os dados foram coletados por meio de câmeras de vídeo equipadas com algoritmos de contagem de pessoas. As      │
│  informações foram registradas em formato JSON e incluem o número de pessoas que entraram e saíram de uma área  │
│  específica, bem como o timestamp correspondente. A análise foi realizada utilizando técnicas estatísticas      │
│  descritivas para resumir e interpretar os dados.                                                               │
│                                                                                                                 │
│  Análise Geral                                                                                                  │
│                                                                                                                 │
│  A análise dos dados revela o seguinte:                                                                         │
│                                                                                                                 │
│  *   **Fluxo Total:** O número total de pessoas que entraram na área monitorada durante o período analisado     │
│  foi de [inserir número total de entradas]. Em contrapartida, o número total de pessoas que saíram foi de       │
│  [inserir número total de saídas]. A diferença entre entradas e saídas pode indicar uma tendência de aumento    │
│  ou diminuição da população na área.                                                                            │
│                                                                                                                 │
│  *   **Picos de Movimentação:** Foram identificados horários de pico de movimentação, nos quais o número de     │
│  entradas e saídas foi significativamente maior do que a média. Esses picos ocorreram em [inserir horários de   │
│  pico]. A identificação desses horários pode ser útil para otimizar a alocação de recursos e melhorar a gestão  │
│  do espaço.                                                                                                     │
│                                                                                                                 │
│  *   **Vale de Movimentação:** Similarmente, foram ide

```text
Relatório de Análise de Contagem de Pessoas

Este relatório apresenta uma análise dos dados de contagem de pessoas obtidos a partir de um sistema de vídeo monitoramento. O objetivo é fornecer uma visão geral do fluxo de pessoas em um determinado local e período, identificando padrões e tendências relevantes.

Metodologia

Os dados foram coletados por meio de câmeras de vídeo equipadas com algoritmos de contagem de pessoas. As informações foram registradas em formato JSON e incluem o número de pessoas que entraram e saíram de uma área específica, bem como o timestamp correspondente. A análise foi realizada utilizando técnicas estatísticas descritivas para resumir e interpretar os dados.

Análise Geral

A análise dos dados revela o seguinte:

*   **Fluxo Total:** O número total de pessoas que entraram na área monitorada durante o período analisado foi de [inserir número total de entradas]. Em contrapartida, o número total de pessoas que saíram foi de [inserir número total de saíd